# Module 3.3 — Magic Methods, Protocols & Context Managers

### Python Mastery · Track 3: Object-Oriented Programming & Design Patterns

Magic methods (also called dunder methods, for their double underscores) let your objects work naturally with Python's built-in syntax: printing, comparison, arithmetic, indexing, iteration, and the `with` statement. This module covers representation, equality and hashing, container behaviour, and the context-manager protocol together with `contextlib`.

**How to use this notebook**

- Read each explanation, then run the code cell beneath it.
- Try the built-in operations (`print`, `==`, `len`, `in`) on your own objects.
- Attempt the exercises before consulting the solutions.

### Learning objectives

After completing this module you will be able to:

1. Implement `__str__` and `__repr__` for readable objects.
2. Define equality with `__eq__` and make objects hashable with `__hash__`.
3. Make objects behave like containers with `__len__`, `__getitem__`, and `__contains__`.
4. Write a context manager with `__enter__` and `__exit__`.
5. Use `contextlib` tools: `contextmanager`, `suppress`, `closing`, and `ExitStack`.

**Prerequisites:** Tracks 1 and 2, and Modules 3.1 to 3.2.

---

## Part 1 · Representation: `__str__` and `__repr__`

Two methods control how an object appears as text. `__repr__` is the unambiguous, developer-facing representation, ideally one that shows how to recreate the object; it is what you see in the interpreter and inside containers. `__str__` is the friendly, user-facing version used by `print` and `str()`. If only `__repr__` is defined, it serves for both.

In [ ]:
class Money:
    def __init__(self, amount, currency="USD"):
        self.amount = amount
        self.currency = currency

    def __repr__(self):
        # Unambiguous: looks like the code to recreate the object.
        return f"Money({self.amount!r}, {self.currency!r})"

    def __str__(self):
        # Friendly: for end users.
        return f"{self.amount:.2f} {self.currency}"

m = Money(19.5, "EUR")
print("print uses __str__:", m)            # 19.50 EUR
print("repr uses __repr__:", repr(m))      # Money(19.5, 'EUR')
print("inside a list, __repr__ is used:", [m])

## Part 2 · Equality and Hashing: `__eq__` and `__hash__`

By default, two distinct objects are considered equal only if they are the very same object. Defining `__eq__` lets you compare by value instead. 

There is an important rule: if you define `__eq__`, Python sets `__hash__` to `None`, making instances **unhashable** (they cannot be used in sets or as dictionary keys) unless you also define `__hash__`. Objects that are equal must have the same hash, so base the hash on the same fields used for equality.

In [ ]:
class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __eq__(self, other):
        if not isinstance(other, Point):
            return NotImplemented        # let Python handle unrelated comparisons
        return (self.x, self.y) == (other.x, other.y)

    def __hash__(self):
        # Equal objects must hash equally; use the same fields as __eq__.
        return hash((self.x, self.y))

    def __repr__(self):
        return f"Point({self.x}, {self.y})"

a, b = Point(1, 2), Point(1, 2)
print("value equality:", a == b)          # True, by value
print("identity:", a is b)                 # False, different objects

# Because __hash__ is defined, Points work in sets and as dict keys.
unique = {Point(0, 0), Point(0, 0), Point(1, 1)}
print("distinct points:", unique)

## Part 3 · Container Behaviour: `__len__`, `__getitem__`, `__contains__`

Implementing a few methods lets your object behave like a built-in container. `__len__` enables `len()`, `__getitem__` enables indexing and slicing (and, on its own, makes the object iterable), and `__contains__` enables the `in` operator.

In [ ]:
class Playlist:
    def __init__(self, songs):
        self._songs = list(songs)

    def __len__(self):
        return len(self._songs)              # enables len(playlist)

    def __getitem__(self, index):
        return self._songs[index]            # enables playlist[i] and slicing

    def __contains__(self, song):
        return song in self._songs           # enables 'song in playlist'

p = Playlist(["a", "b", "c", "d"])
print("length:", len(p))
print("first:", p[0], "| slice:", p[1:3])
print("'c' present:", "c" in p)

# __getitem__ also makes the object iterable, so a for loop works:
for song in p:
    print("playing", song)

## Part 4 · The Context-Manager Protocol: `__enter__` and `__exit__`

The `with` statement guarantees setup and cleanup around a block, even if an error occurs. Any object that implements `__enter__` (run on entry; its return value is bound by `as`) and `__exit__` (run on exit, including after an exception) is a context manager. This is how files close themselves, and you can give your own resources the same safety.

In [ ]:
class Timer:
    """A context manager that reports how long its block took."""
    def __enter__(self):
        self.events = []
        self.events.append("entered")
        return self                          # bound to the name after 'as'

    def __exit__(self, exc_type, exc_value, traceback):
        self.events.append("exited")
        # Returning False (or None) lets any exception propagate; True would suppress it.
        return False

with Timer() as t:
    t.events.append("inside the block")

print("sequence of events:", t.events)

# __exit__ runs even when the block raises, ensuring cleanup.
class Guard:
    def __enter__(self):
        print("acquire resource")
        return self
    def __exit__(self, exc_type, exc_value, tb):
        print("release resource (always runs)")
        return False

try:
    with Guard():
        print("working...")
        raise ValueError("something failed")
except ValueError as e:
    print("caught after cleanup:", e)

## Part 5 · The `contextlib` Toolkit

Writing a full class for a context manager is sometimes more than you need. The `contextlib` module offers lighter tools:

- `@contextmanager` turns a generator into a context manager: code before `yield` is setup, code after is cleanup.
- `suppress(...)` ignores specified exceptions within a block.
- `closing(...)` calls an object's `close()` method on exit.
- `ExitStack` manages a dynamic number of context managers together.

In [ ]:
from contextlib import contextmanager, suppress

@contextmanager
def tag(name):
    print(f"<{name}>", end="")      # setup
    yield                            # the body of the with-block runs here
    print(f"</{name}>")              # cleanup

with tag("b"):
    print("bold text", end="")

# suppress: ignore a specific error without a try/except.
with suppress(ZeroDivisionError):
    result = 1 / 0
    print("this line is skipped")
print("continued past the suppressed error")

In [ ]:
from contextlib import ExitStack

# ExitStack manages a variable number of context managers, all closed correctly.
class Resource:
    def __init__(self, name):
        self.name = name
    def __enter__(self):
        print("open", self.name)
        return self
    def __exit__(self, *exc):
        print("close", self.name)
        return False

names = ["db", "cache", "file"]
with ExitStack() as stack:
    resources = [stack.enter_context(Resource(n)) for n in names]
    print("using:", [r.name for r in resources])
# All three are closed in reverse order as the stack unwinds.

---

## Worked Examples

| Examples | Level |
|---|---|
| 1 and 2 | Easy |
| 3 and 4 | Medium |
| 5 and 6 | Difficult |

### Example 1 — Readable objects with __repr__ and __str__ (Easy)

In [ ]:
class Color:
    def __init__(self, r, g, b):
        self.r, self.g, self.b = r, g, b
    def __repr__(self):
        return f"Color({self.r}, {self.g}, {self.b})"
    def __str__(self):
        return f"#{self.r:02x}{self.g:02x}{self.b:02x}"

c = Color(255, 128, 0)
print("friendly:", c)        # hex string
print("developer:", repr(c)) # constructor form

### Example 2 — Value equality (Easy)

In [ ]:
class Version:
    def __init__(self, major, minor):
        self.major, self.minor = major, minor
    def __eq__(self, other):
        return (self.major, self.minor) == (other.major, other.minor)
    def __hash__(self):
        return hash((self.major, self.minor))

print(Version(1, 4) == Version(1, 4))   # True
print(Version(1, 4) == Version(2, 0))   # False

### Example 3 — A container-like class (Medium)

In [ ]:
class Inventory:
    def __init__(self):
        self._items = {}

    def add(self, name, quantity):
        self._items[name] = self._items.get(name, 0) + quantity

    def __len__(self):
        return len(self._items)

    def __getitem__(self, name):
        return self._items[name]

    def __contains__(self, name):
        return name in self._items

inv = Inventory()
inv.add("apple", 3)
inv.add("pear", 5)
print("kinds of item:", len(inv))
print("apples:", inv["apple"])
print("has pear:", "pear" in inv)

### Example 4 — A context manager with @contextmanager (Medium)

In [ ]:
from contextlib import contextmanager

@contextmanager
def section(title):
    print(f"=== {title} ===")     # setup
    try:
        yield                      # the block runs here
    finally:
        print("=== end ===")       # cleanup, even on error

with section("Report"):
    print("line one")
    print("line two")

### Example 5 — Arithmetic via magic methods (Difficult)

In [ ]:
class Vector:
    def __init__(self, x, y):
        self.x, self.y = x, y

    def __add__(self, other):              # enables v1 + v2
        return Vector(self.x + other.x, self.y + other.y)

    def __mul__(self, scalar):             # enables v * number
        return Vector(self.x * scalar, self.y * scalar)

    def __eq__(self, other):
        return (self.x, self.y) == (other.x, other.y)

    def __repr__(self):
        return f"Vector({self.x}, {self.y})"

v = Vector(1, 2) + Vector(3, 4)
print("sum:", v)
print("scaled:", v * 3)
print("equal check:", Vector(2, 2) == Vector(2, 2))

### Example 6 — A managed resource with full protocol and ExitStack (Difficult)

In [ ]:
from contextlib import ExitStack

class Connection:
    """A pretend connection that records its lifecycle."""
    log = []
    def __init__(self, name):
        self.name = name
    def __enter__(self):
        Connection.log.append(f"open {self.name}")
        return self
    def query(self, q):
        return f"{self.name}: result of {q!r}"
    def __exit__(self, exc_type, exc_value, tb):
        Connection.log.append(f"close {self.name}")
        return False

# Open several connections together; all close in reverse order at the end.
with ExitStack() as stack:
    conns = [stack.enter_context(Connection(n)) for n in ["primary", "replica"]]
    for conn in conns:
        print(conn.query("SELECT 1"))

print("lifecycle:", Connection.log)

---

## Exercises

**Exercise 1 (Easy).** Define a class `Temperature` storing `celsius`, with a `__str__` returning `"25.0 C"` and a `__repr__` returning `"Temperature(25.0)"`. Demonstrate both.

In [ ]:
# Your solution here


**Exercise 2 (Medium).** Define a class `Coordinate` with `x` and `y`, value equality via `__eq__`, and `__hash__` so instances can go in a set. Build a set with a duplicate and show only distinct coordinates remain.

In [ ]:
# Your solution here


**Exercise 3 (Medium).** Define a class `Bag` that wraps a list and supports `len()`, indexing with `[]`, and the `in` operator by implementing `__len__`, `__getitem__`, and `__contains__`. Demonstrate all three.

In [ ]:
# Your solution here


**Exercise 4 (Medium).** Using `@contextmanager`, write a context manager `banner(text)` that prints a line of equals signs before and after the block. Use it around two print statements.

In [ ]:
# Your solution here


**Exercise 5 (Difficult).** Define a class `Stopwatch` as a context manager that, on entry, records a fixed start value of 0 and, on exit, appends the strings `"start"` and `"stop"` to an instance list `marks`. After the block, print `marks`. (Use fixed values rather than the real clock so the output is reproducible.)

In [ ]:
# Your solution here


---

## Solutions

**Solution 1**

In [ ]:
class Temperature:
    def __init__(self, celsius):
        self.celsius = celsius
    def __str__(self):
        return f"{self.celsius:.1f} C"
    def __repr__(self):
        return f"Temperature({self.celsius})"

t = Temperature(25.0)
print(str(t))
print(repr(t))

**Solution 2**

In [ ]:
class Coordinate:
    def __init__(self, x, y):
        self.x, self.y = x, y
    def __eq__(self, other):
        return (self.x, self.y) == (other.x, other.y)
    def __hash__(self):
        return hash((self.x, self.y))
    def __repr__(self):
        return f"Coordinate({self.x}, {self.y})"

s = {Coordinate(1, 1), Coordinate(1, 1), Coordinate(2, 3)}
print(s)

**Solution 3**

In [ ]:
class Bag:
    def __init__(self, items):
        self._items = list(items)
    def __len__(self):
        return len(self._items)
    def __getitem__(self, i):
        return self._items[i]
    def __contains__(self, value):
        return value in self._items

b = Bag(["x", "y", "z"])
print(len(b), b[0], "y" in b)

**Solution 4**

In [ ]:
from contextlib import contextmanager

@contextmanager
def banner(text):
    print("=" * 20)
    yield
    print("=" * 20)

with banner("hi"):
    print("first")
    print("second")

**Solution 5**

In [ ]:
class Stopwatch:
    def __enter__(self):
        self.marks = []
        self.start = 0
        self.marks.append("start")
        return self
    def __exit__(self, exc_type, exc_value, tb):
        self.marks.append("stop")
        return False

with Stopwatch() as sw:
    pass
print(sw.marks)

---

## Summary and Key Points

- `__repr__` is the unambiguous developer representation (used in the interpreter and containers); `__str__` is the friendly version used by `print`. `__repr__` serves for both if `__str__` is absent.
- `__eq__` enables value comparison; defining it removes hashing, so also define `__hash__` (over the same fields) to keep objects usable in sets and as dict keys.
- `__len__`, `__getitem__`, and `__contains__` give an object container behaviour; `__getitem__` alone also makes it iterable.
- A context manager implements `__enter__` (setup; value bound by `as`) and `__exit__` (cleanup; runs even on error). Returning a true value from `__exit__` suppresses the exception.
- `contextlib` provides lighter tools: `@contextmanager` (generator-based), `suppress` (ignore chosen errors), `closing` (call `close`), and `ExitStack` (manage many context managers).

### Next module: 3.4 — Properties, Descriptors & Dataclasses

The next module covers controlling attribute access with properties, the descriptor protocol that powers them, and writing concise data-holding classes with dataclasses.